In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/content/retail_store_sales.csv")
print(df.shape)
df.head()
df.info()

(12575, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [ ]:
print(df.isnull().sum())  # before

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [ ]:
print("Duplicate rows:",df.duplicated().sum())
print("Duplicate Transaction IDs:",df["Transaction ID"].duplicated().sum())

Duplicate rows: 0
Duplicate Transaction IDs: 0


In [ ]:
print(df["Category"].unique())
print(df["Payment Method"].unique())
print(df["Location"].unique())
print(df["Discount Applied"].unique())
print(df["Item"].nunique())

['Patisserie' 'Milk Products' 'Butchers' 'Beverages' 'Food' 'Furniture'
 'Electric household essentials' 'Computers and electric accessories']
['Digital Wallet' 'Credit Card' 'Cash']
['Online' 'In-store']
[True False nan]
200


In [ ]:
check = df.dropna(subset=["Price Per Unit", "Quantity", "Total Spent"])
print("Total Spent mismatch:", (~np.isclose(check["Price Per Unit"] * check["Quantity"], check["Total Spent"])).sum())

print(df[["Price Per Unit", "Quantity", "Total Spent"]].describe())

print("Invalid dates:", pd.to_datetime(df["Transaction Date"], errors="coerce").isna().sum())

Total Spent mismatch: 0
       Price Per Unit      Quantity   Total Spent
count    11966.000000  11971.000000  11971.000000
mean        23.365912      5.536380    129.652577
std         10.743519      2.857883     94.750697
min          5.000000      1.000000      5.000000
25%         14.000000      3.000000     51.000000
50%         23.000000      6.000000    108.500000
75%         33.500000      8.000000    192.000000
max         41.000000     10.000000    410.000000
Invalid dates: 0


In [ ]:
df_raw = df.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns.tolist())

['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


In [ ]:
before = len(df)

df = df.drop_duplicates()
df = df.drop_duplicates(subset="transaction_id")

print("Rows before:", before)
print("Rows after:", len(df))
print("Duplicates removed:", before - len(df))

Rows before: 12575
Rows after: 12575
Duplicates removed: 0


In [ ]:
for col in ["price_per_unit", "quantity", "total_spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")

df["discount_applied"] = df["discount_applied"].astype("boolean")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    12575 non-null  object        
 1   customer_id       12575 non-null  object        
 2   category          12575 non-null  object        
 3   item              11362 non-null  object        
 4   price_per_unit    11966 non-null  float64       
 5   quantity          11971 non-null  float64       
 6   total_spent       11971 non-null  float64       
 7   payment_method    12575 non-null  object        
 8   location          12575 non-null  object        
 9   transaction_date  12575 non-null  datetime64[ns]
 10  discount_applied  8376 non-null   boolean       
dtypes: boolean(1), datetime64[ns](1), float64(3), object(6)
memory usage: 1007.1+ KB


In [ ]:
# 1. Total Spent = Price x Quantity
m = df["total_spent"].isna() & df["price_per_unit"].notna() & df["quantity"].notna()
df.loc[m, "total_spent"] = df["price_per_unit"] * df["quantity"]

# 2. Price = Total / Quantity
m = df["price_per_unit"].isna() & df["total_spent"].notna() & df["quantity"].notna()
df.loc[m, "price_per_unit"] = df["total_spent"] / df["quantity"]

# 3. Quantity = Total / Price
m = df["quantity"].isna() & df["total_spent"].notna() & df["price_per_unit"].notna()
df.loc[m, "quantity"] = df["total_spent"] / df["price_per_unit"]

# 4. Item ka naam andaaze se nahi bhar sakte
df["item"] = df["item"].fillna("Unknown")

print(df.isnull().sum())

transaction_id         0
customer_id            0
category               0
item                   0
price_per_unit         0
quantity             604
total_spent          604
payment_method         0
location               0
transaction_date       0
discount_applied    4199
dtype: int64


In [ ]:
both_missing = (df["quantity"].isna() & df["total_spent"].isna()).sum()
print("Rows jahan quantity aur total dono khaali hain:", both_missing)

before = len(df)
df = df.dropna(subset=["quantity", "total_spent"])
df["quantity"] = df["quantity"].round().astype(int)

print("Rows dropped:", before - len(df))
print("Final shape:", df.shape)
print(df.isnull().sum())

Rows jahan quantity aur total dono khaali hain: 604
Rows dropped: 604
Final shape: (11971, 11)
transaction_id         0
customer_id            0
category               0
item                   0
price_per_unit         0
quantity               0
total_spent            0
payment_method         0
location               0
transaction_date       0
discount_applied    3988
dtype: int64


In [ ]:
print("BEFORE shape:", df_raw.shape)
print("AFTER shape:", df.shape)

print("\nDuplicate rows:", df.duplicated().sum())
print("Total missing values before:", df_raw.isnull().sum().sum())
print("Total missing values after:", df.isnull().sum().sum())

print("\nData types after cleaning:")
print(df.dtypes)

df.to_csv("cleaned_data.csv", index=False)

BEFORE shape: (12575, 11)
AFTER shape: (11971, 11)

Duplicate rows: 0
Total missing values before: 7229
Total missing values after: 3988

Data types after cleaning:
transaction_id              object
customer_id                 object
category                    object
item                        object
price_per_unit             float64
quantity                     int64
total_spent                float64
payment_method              object
location                    object
transaction_date    datetime64[ns]
discount_applied           boolean
dtype: object


In [ ]:
from google.colab import files
files.download("cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>